## Analyzing Quality of Clustering

TODO:

 - massive convertation of references to PMIDs

Lotte et al., 2007 - no references stored in Pubmed, the same also for Lotte et al., 2018 :(

In [ ]:
"""
Closed-loop brain training: the science of neurofeedback (2017)

Two distinct areas of clinical applications are discussed: 
 - ADHD (attention deficit hyperactivity disorder)
 - rehabilitation in stroke.
"""
CLUSTERS = {
    'neurofeedback': {
        'adhd': [7786929, 19712709, 19207632, 23665196, 24399101, 26748531],
        'Stroke': [22401758, 23494615, 24217509]
    }
}

In [ ]:
import gzip
import itertools
import json

from collections import Counter

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig, AnalyzerSettings
from pysrc.papers.db.loaders import Loaders
from pysrc.papers.utils import vectorize_corpus, compute_tfidf

In [ ]:
PUBTRENDS_CONFIG = PubtrendsConfig(test=False)


def reload_exported_analyzer(path_to_archive, source='Pubmed'):
    """
    Load analysis data from json.gz archive generated by PubTrends.
    """
    with gzip.open(path_to_archive, 'rt', encoding='UTF-8') as zipfile:
        data = json.load(zipfile)

    loader, url_prefix = Loaders.get_loader_and_url_prefix(source, PUBTRENDS_CONFIG)
    analyzer = KeyPaperAnalyzer(loader, PUBTRENDS_CONFIG)
    analyzer.init(data)
    
    # Fill other fields (TODO: move to init)
    analyzer.ids = set(analyzer.df['id'])
    analyzer.n_papers = len(analyzer.ids)
    analyzer.pub_types = list(set(analyzer.df['type']))
    analyzer.query = 'restored from PubTrends export'
    
    PUB_DF_COLUMNS = ['id', 'title', 'abstract', 'year', 'type', 'keywords', 'mesh', 'doi', 'aux']
    analyzer.pub_df = analyzer.df[PUB_DF_COLUMNS]
    
    analyzer.components = set(analyzer.df['comp'].unique())
    if -1 in analyzer.components:
        analyzer.components.remove(-1)
        
    settings = AnalyzerSettings()
    analyzer.corpus_ngrams, analyzer.corpus_counts = \
        vectorize_corpus(analyzer.pub_df,
                         max_features=settings.VECTOR_WORDS, n_gram=settings.VECTOR_NGRAMS,
                         min_df=settings.VECTOR_MIN_DF, max_df=settings.VECTOR_MAX_DF)
    tfidf = compute_tfidf(analyzer.corpus_counts)
    analyzer.texts_similarity = analyzer.analyze_texts_similarity(analyzer.pub_df, tfidf,
                                                                  settings.SIMILARITY_TEXT_MIN,
                                                                  settings.SIMILARITY_TEXT_CITATION_N)
    
    return analyzer

In [ ]:
PATH_TO_ZIP = '/home/willenjoy/work/pubtrends/local/pubtrends_export/64.json.gz'
REVIEW_ID = '28003656'

analyzer = reload_exported_analyzer(PATH_TO_ZIP)

In [ ]:
len(analyzer.components) # Equals to 11 in the current PubTrends setup

## Number of components

In [ ]:
param_grid = {
    'SIMILARITY_BIBLIOGRAPHIC_COUPLING': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_COCITATION': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_CITATION': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_TEXT_CITATION': [0.01, 0.1, 1, 10, 100]
}

In [ ]:
def settings_of_interest(analyzer_settings):
    return (
        analyzer_settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
        analyzer_settings.SIMILARITY_COCITATION,
        analyzer_settings.SIMILARITY_CITATION,
        analyzer_settings.SIMILARITY_TEXT_CITATION,
    )

In [ ]:
baseline_params = dict(
    TOPIC_MIN_SIZE=0,
    TOPICS_MAX_NUMBER=500
)

In [ ]:
param_names = param_grid.keys()
result = {}

# Iterate over param grid
for param_values in itertools.product(*param_grid.values()):
    params = {k: v for k, v in zip(param_names, param_values)}
    for k, v in baseline_params.items():
        params[k] = v
    
    settings = AnalyzerSettings(**params)
    analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                            settings=settings, load_data=False, compute_topic_descriptions=False)

    soi = settings_of_interest(settings)
    comps = len(analyzer.components)
    sizes = Counter(analyzer.partition.values())
    max_size = max(sizes.values())
    min_size = min(sizes.values())
    print(f'{soi} - {comps} components, max size = {max_size}, min_size = {min_size}')
    result[soi] = (comps, max_size, min_size)

## Experiments with Clustering

**Ground Truth**

References split into clusters according to their occurrences in text sections.

In [ ]:
"""
Closed-loop brain training: the science of neurofeedback (2017)

Clustering based on occurrence in different sections.
Calculated below in the 'Grobid Reference Extraction' section.
"""

print(SECTION_CLUSTERING)

In [ ]:
print(APPLICATION_CLUSTERING)

### How to evaluate clustering?

**Rand index** (0 - bad, 1 - good)

Each pair of papers belongs to either same or different cluster. Rand index denotes accuracy of pair classification in terms of same/different cluster compared to ground truth.

**Adjusted Rand index** (-1 - bad, 0 - random, 1 - good)

A version of Rand index adjusted against random clustering.

**TODO: add other metrics and check their correlation: https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation**

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, contingency_matrix, pair_confusion_matrix, v_measure_score

In [ ]:
def get_clustering(analyzer, target_ids):
    data = analyzer.df[analyzer.df['id'].isin(target_ids)]
    return {k: v for k, v in zip(data['id'], data['comp'])}

In [ ]:
def evaluate_clustering(analyzer, ground_truth):
    actual_clustering = get_clustering(analyzer, ground_truth.keys())
    
    # Align clusterings
    labels_true = []
    labels_pred = []

    for pmid in actual_clustering:
        labels_true.append(ground_truth[pmid])
        labels_pred.append(actual_clustering[pmid])
        
    # Evaluate (Rand index, contingency matrix)
    rand_score = adjusted_rand_score(labels_true, labels_pred)
    cm = contingency_matrix(labels_true, labels_pred) 
    
    return score, cm

## Current PubTrends Results

In [ ]:
analyzer = reload_exported_analyzer(PATH_TO_ZIP)

In [ ]:
evaluate_clustering(analyzer, SECTION_CLUSTERING)

In [ ]:
evaluate_clustering(analyzer, APPLICATION_CLUSTERING)

## Grid Search

In [ ]:
param_grid = {
    'SIMILARITY_BIBLIOGRAPHIC_COUPLING': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_COCITATION': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_CITATION': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_TEXT_CITATION': [0.01, 0.1, 1, 10, 100]
}

In [ ]:
def settings_of_interest(analyzer_settings):
    return (
        analyzer_settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
        analyzer_settings.SIMILARITY_COCITATION,
        analyzer_settings.SIMILARITY_CITATION,
        analyzer_settings.SIMILARITY_TEXT_CITATION,
    )

In [ ]:
baseline_params = dict(
    TOPIC_MIN_SIZE=0,
    TOPICS_MAX_NUMBER=500
)

In [ ]:
param_names = param_grid.keys()
best_score = 0
best_cm = None

# Iterate over param grid
for param_values in itertools.product(*param_grid.values()):
    params = {k: v for k, v in zip(param_names, param_values)}
    for k, v in baseline_params.items():
        params[k] = v
    
    settings = AnalyzerSettings(**params)
    analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                            settings=settings, load_data=False, compute_topic_descriptions=False)

    soi = settings_of_interest(settings)
    score, cm = evaluate_clustering(analyzer, SECTION_CLUSTERING)
    new_best = False
    if score > best_score:
        best_score = score
        best_cm = cm
        new_best = True
        
    print(f"{'BEST' if new_best else ''} {soi} - score {score}")

In [ ]:
# Adjusted Rand index > 0.2

BEST_SOI = (0.01, 0.1, 100, 100)
BEST_SCORE = 0.2153548037588293
SOI_GOOD = [
    (0.01, 0.01, 10, 10),
    (0.1, 0.01, 10, 10),
    (0.1, 0.01, 100, 100),
    (0.1, 0.1, 100, 100),
    (1, 0.1, 100, 100)
]

In [ ]:
params = {k: v for k, v in zip(param_names, BEST_SOI)}
for k, v in baseline_params.items():
    params[k] = v

settings = AnalyzerSettings(**params)
analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                        settings=settings, load_data=False, compute_topic_descriptions=False)

soi = settings_of_interest(settings)
score, cm = evaluate_clustering(analyzer, SECTION_CLUSTERING)

print(score)
print(cm)

APPLICATION_CLUSTERING

In [ ]:
param_names = param_grid.keys()
best_soi = None
best_score = 0
best_cm = None

# Iterate over param grid
for param_values in itertools.product(*param_grid.values()):
    params = {k: v for k, v in zip(param_names, param_values)}
    for k, v in baseline_params.items():
        params[k] = v
    
    settings = AnalyzerSettings(**params)
    analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                            settings=settings, load_data=False, compute_topic_descriptions=False)

    soi = settings_of_interest(settings)
    score, cm = evaluate_clustering(analyzer, APPLICATION_CLUSTERING)
    new_best = False
    if score > best_score:
        best_soi = soi
        best_score = score
        best_cm = cm
        new_best = True
        
    print(f"{'BEST' if new_best else ''} {soi} - score {score}")

In [ ]:
print(best_score)
print(best_soi)
print(best_cm)

In [ ]:
params = {k: v for k, v in zip(param_names, best_soi)}
for k, v in baseline_params.items():
    params[k] = v

settings = AnalyzerSettings(**params)
analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                        settings=settings, load_data=False, compute_topic_descriptions=False)

print(analyzer.components)

## OLD - Experiments with Manual Clustering

In [ ]:
marked_refs = []
for refs in CLUSTERS.values():
    marked_refs.extend([str(r) for r in refs])

In [ ]:
marked_refs

In [ ]:
for _, row in analyzer.df_kwd.iterrows():
    keywords = list(map(lambda el: el[0], row['kwd']))
    print(row['comp'] + 1, ",".join(keywords))

In [ ]:
analyzer.df[analyzer.df['id'].isin(marked_refs)][['id', 'title', 'comp']]

In [ ]:
PATH_TO_ZIP_ZOOMED = '/home/willenjoy/work/pubtrends/local/pubtrends_export/Neurofeedback-Ros-2017_subtopic1_detailed.zip'

zoomed_analyzer = reload_exported_analyzer(PATH_TO_ZIP_ZOOMED)

In [ ]:
for _, row in zoomed_analyzer.df_kwd.iterrows():
    keywords = list(map(lambda el: el[0], row['kwd']))
    print(row['comp'] + 1, ",".join(keywords))

In [ ]:
zoomed_analyzer.df[zoomed_analyzer.df['id'].isin(marked_refs)][['id', 'title', 'comp']]

In [ ]:
NEUROFEEDBACK_CLUSTERS

## Grobid Reference Extraction

In [ ]:
import requests

from bs4 import BeautifulSoup

In [ ]:
GROBID_API_URL = 'http://localhost:8070/api/processReferences'
PDF_FILE = '/home/willenjoy/Downloads/nrn.2016.164.pdf'
XML_FILE = '/home/willenjoy/Downloads/nrn.2016.164-refs.tei.xml'

In [ ]:
"""
TODO: retry

https://github.com/kermitt2/grobid_client_python/blob/master/grobid_client.py
"""

def extract_references(pdf_file):
    files = {
        'input': {
            pdf_file,
            open(pdf_file, 'rb'),
            'application/pdf',
            {'Expires': '0'}
        }
    }
    
    res, status = requests.post(
        GROBID_API_URL,
        files=files,
        headers={'Accept': 'application/xml'}
    )
    
    return res, status

In [ ]:
# res, status = extract_references(PDF_FILE)

In [ ]:
with open(XML_FILE, 'r') as refs_xml:
    soup = BeautifulSoup(refs_xml, 'xml')

In [ ]:
refs = soup.select('biblStruct')

In [ ]:
len(refs)

In [ ]:
pubmed_ref_ids = list(analyzer.citations_graph.successors(REVIEW_ID))

In [ ]:
pubmed_data = analyzer.df[analyzer.df['id'].isin(pubmed_ref_ids)]
pubmed_ref_search_index = {v.lower(): k for k, v in zip(pubmed_data['id'], pubmed_data['title'])}
pubmed_ref_titles = {k: v for k, v in zip(pubmed_data['id'], pubmed_data['title'])}

In [ ]:
pubmed_ref_titles['20692351'].lower() in pubmed_ref_search_index

In [ ]:
found = 0
num2pmid = {}

for i, ref in enumerate(refs):
    if ref.analytic and ref.analytic.title:
        ref_title = ref.analytic.title.text
        if ref_title.lower() in pubmed_refs:
            pmid = pubmed_refs[ref_title.lower()]
            print(i + 1, ref_title, pmid)
            num2pmid[i + 1] = pmid
            found += 1
        
print(f'Found {found} references')

In [ ]:
num2pmid

In [ ]:
"""
FIXME:
 - how to resolve multiple occurrences in text? (currently use first occurrence)
 - numbers shift in current enumerate implementation, use ref ids in XML (after 78 +1, after 120 +3)
"""

SECTION_CLUSTERING_RAW = {
    'Neural specificity and plasticity': {
        'The neural substrates of self-regulation': [3, 4, 6] + list(range(11, 44)),
        'Neural plasticity and specificity': list(range(44, 68)),
    },
    'Neural mechanisms of self-regulation': {
        'Factors influencing neurofeedback learning': list(range(70, 86)),
        'Neural substrates of self-regulation': list(range(86, 94)),
    },
    'Clinical applications of neurofeedback': {
        'Attention deficit hyperactivity disorder': list(range(94, 137)),
        'Rehabilitation in stroke': list(range(138, 156)),
    }
}

In [ ]:
clusters = []

for section_refs in SECTION_CLUSTERING_RAW.values():
    for subsection, raw_refs in section_refs.items():
        cluster = []
        for ref in raw_refs:
            if ref in num2pmid:
                cluster.append(num2pmid[ref])
        clusters.append(cluster)

In [ ]:
for i, cluster in enumerate(clusters):
    print(f'Cluster {i + 1}')
    print(analyzer.df[analyzer.df['id'].isin(cluster)][['id', 'title']])

In [ ]:
SECTION_CLUSTERING = {}

for i, cluster in enumerate(clusters):
    for ref in cluster:
        SECTION_CLUSTERING[ref] = i

In [ ]:
SECTION_CLUSTERING

In [ ]:
APPLICATION_CLUSTERING = {}

for ref in clusters[4]:
    APPLICATION_CLUSTERING[ref] = 0
for ref in clusters[5]:
    APPLICATION_CLUSTERING[ref] = 1

In [ ]:
APPLICATION_CLUSTERING

In [ ]:
analyzer.similarity_graph.edges(data=True)